# Scotoma: train and benchmark a redaction model

Runtime → Change runtime type → GPU (a T4 is enough for the small backbones).

First run end to end on 2026-10-04 (Colab L4, about 70 minutes for training). Results and fixes: `bench/results/README.md`.

What it does: builds the scorer, generates synthetic data, mixes in an open PII corpus, fine-tunes a token classifier on Scotoma's categories, exports the ONNX model the app loads, and scores it against the rules and a public baseline on held-out data.

In [ ]:
# 1. Upload scotoma.zip (the project folder, zipped)
from google.colab import files
import os, zipfile
if not os.path.isdir('scotoma'):
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('.')
%cd scotoma

In [ ]:
# 2. Dependencies and the scoring CLI (Rust build, about 3 minutes)
# optimum is no longer needed (its exporter broke on Colab). transformers is pinned to 4.x: the verified run used 4.57.6, 5.x is untested here.
!pip -q install "transformers>=4.57,<5" onnx onnxruntime onnxscript datasets
!curl -sSf https://sh.rustup.rs | sh -s -- -y -q
!. $HOME/.cargo/env && cargo build --release -p scotoma 2>&1 | tail -2

## Data

Three sources, kept separate on purpose:

- **Templates, universe A** → training. **Templates, universe B** → test. The two share no names, places, organisations, templates or background sentences.
- **An open PII corpus** (Nemotron-PII) → training and a second test set. This supplies the natural, varied background text the templates lack. Without it a model learns to redact anything unfamiliar; the smoke test did exactly that (99.9% recall while redacting 67% of ordinary text).
- **LLM-written documents** per domain (`bench/llm_generate.py`) → add these for law, tax and HR, where no open corpus exists. Generate them wherever you have an LLM endpoint and upload the JSONL.

In [ ]:
!mkdir -p bench/data
!python bench/generate.py --universe A --n 30000 --seed 7 --out bench/data/train_tpl.jsonl
!python bench/generate.py --universe A --n 2000  --seed 8 --out bench/data/dev_tpl.jsonl
!python bench/generate.py --universe B --n 3000  --seed 1 --out bench/data/test_tpl.jsonl

# Open corpus → Scotoma categories. Check the 'first row' line each prints.
!python scripts/prep_dataset.py nvidia/Nemotron-PII --split train --limit 50000 --out bench/data/nemo_train_raw.jsonl
!python scripts/prep_dataset.py nvidia/Nemotron-PII --split test  --limit 4000  --out bench/data/nemo_test_raw.jsonl
!./target/release/scotoma convert bench/data/nemo_train_raw.jsonl bench/data/nemo_train.jsonl
!./target/release/scotoma convert bench/data/nemo_test_raw.jsonl  bench/data/nemo_test_all.jsonl
!head -n 2000 bench/data/nemo_test_all.jsonl > bench/data/nemo_dev.jsonl
!tail -n 2000 bench/data/nemo_test_all.jsonl > bench/data/nemo_test.jsonl

## Train

`--base` can be any encoder with a fast tokenizer. `microsoft/deberta-v3-small` is the same backbone as the OpenMed baseline; `answerdotai/ModernBERT-base` is larger and handles long documents. `--o-weight 0.7` trades some precision for recall.

In [ ]:
!python -u train/train.py \
  --train bench/data/train_tpl.jsonl bench/data/nemo_train.jsonl \
  --dev bench/data/dev_tpl.jsonl bench/data/nemo_dev.jsonl \
  --base microsoft/deberta-v3-small --out models/scotoma-v0 --epochs 3 --batch 32 --lr 5e-5

## Score

Baseline: the public OpenMed clinical PII model, exported and scored by the same code. Read **over-redaction** next to recall: high recall with high over-redaction is not a win.

Nemotron-PII is in both models' training distribution, so that table is a ceiling. The template test is out-of-distribution for the baseline and in-family for ours, so a win there is expected and proves little. The comparison that would mean something is a test set neither model's training data resembles: LLM-written documents from a different model, or i2b2.

In [ ]:
!python scripts/fetch_model.py --out models/openmed
!python bench/run.py bench/data/test_tpl.jsonl --strict --out bench/results/templates \
   --system rules --system ours=models/scotoma-v0 --system rules+ours=models/scotoma-v0 --system openmed=models/openmed --system rules+openmed=models/openmed
!python bench/run.py bench/data/nemo_test.jsonl --out bench/results/nemotron \
   --system rules --system ours=models/scotoma-v0 --system rules+ours=models/scotoma-v0 --system openmed=models/openmed --system rules+openmed=models/openmed
!cat bench/results/templates/results.md bench/results/nemotron/results.md

In [ ]:
# Download the model (drop the folder into app/src-tauri/models/default) and the results
!rm -f models/scotoma-v0/model_fp32.onnx && cd models && zip -qr ../scotoma-v0.zip scotoma-v0 && cd .. && zip -qr results.zip bench/results
files.download('scotoma-v0.zip'); files.download('results.zip')